# Importar Librerías

Se importan las librerías esenciales para facilitar el análisis que abarca la carga de datos, la evaluación estadística, la visualización, la transformación de datos, la fusión y la unión.

In [1]:
pip install kagglehub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import kagglehub
import os
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
from scipy.stats import kruskal
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

c:\Users\david\miniconda3\envs\ml_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Lectura de Datos

Para la carga inicial del dataset, se utilizó la API de Kaggle Hub mediante Python, permitiendo una descarga automatizada y eficiente de los datos más recientes. El proceso consistió en:

1. **Descarga del Dataset**: Se obtuvo la versión actualizada del dataset 'Craigslist CarsTrucks Data' directamente desde Kaggle, almacenándose localmente en una ruta específica del sistema.

2. **Verificación de Archivos**: Se listaron los archivos descargados para confirmar la presencia del archivo principal (`vehicles.csv`).

3. **Carga en DataFrame**: El archivo CSV se leyó utilizando Pandas, asegurando una estructura lista para el análisis posterior.


In [3]:
path = kagglehub.dataset_download("austinreese/craigslist-carstrucks-data")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\david\.cache\kagglehub\datasets\austinreese\craigslist-carstrucks-data\versions\10


In [4]:
downloaded_files = os.listdir(path)

print("Archivos disponibles:", downloaded_files)

Archivos disponibles: ['vehicles.csv']


In [5]:
csv_path = os.path.join(path, "vehicles.csv") 

df = pd.read_csv(csv_path)

---

Se emplea el método `df.info()` para evaluar la estructura y calidad inicial del dataset. Este análisis revela información sobre la composición de los datos, incluyendo el número total de registros (filas) y variables (columnas), los tipos de datos asociados a cada campo (como objetos, enteros o valores flotantes), y la cantidad de valores no nulos por columna.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 26 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            426880 non-null  int64  
 1   url           426880 non-null  object 
 2   region        426880 non-null  object 
 3   region_url    426880 non-null  object 
 4   price         426880 non-null  int64  
 5   year          425675 non-null  float64
 6   manufacturer  409234 non-null  object 
 7   model         421603 non-null  object 
 8   condition     252776 non-null  object 
 9   cylinders     249202 non-null  object 
 10  fuel          423867 non-null  object 
 11  odometer      422480 non-null  float64
 12  title_status  418638 non-null  object 
 13  transmission  424324 non-null  object 
 14  VIN           265838 non-null  object 
 15  drive         296313 non-null  object 
 16  size          120519 non-null  object 
 17  type          334022 non-null  object 
 18  pain

**Interpretación del Dataset**

1. **Volumen y Dimensiones**:

    - El conjunto de datos contiene 426,880 registros (filas) y 26 variables (columnas), lo que representa un volumen considerable de información sobre vehículos.

2. **Tipos de Datos**:

    - Predominan variables categóricas/cualitativas (19 columnas tipo object).

    - Variables numéricas incluyen 4 columnas float64 (`year`, `odometer`, `lat`, `long`) y 2 int64 (`id` y `price`). La variable `county` es tomada como float64 pero la columna esta totalmente vacía.

    - La columna `posting_date` está como texto (object), requiriendo conversión a datetime.

El dataset presenta un volumen masivo (426,880 registros) con amplia dimensionalidad (26 variables), ofreciendo un amplio potencial para análisis multivariados. Sin embargo, existe un desequilibrio significativo en la composición de datos, con predominio de variables categóricas (19 columnas) sobre numéricas (7 columnas), lo que implica desafíos en el preprocesamiento para modelos predictivos. La combinación de tamaño sustancial y diversidad de variables proporciona una base sólida para análisis exhaustivos, aunque requiere transformaciones intensivas en el manejo de fechas y limpieza de columnas vacías para maximizar su utilidad analítica.

## Valores Nulos

Para cuantificar la magnitud de datos ausentes en cada columna, se calculó el porcentaje de valores nulos mediante `df.isnull().mean() * 100`. Este indicador permite identificar variables críticas que requieren intervención prioritaria en el proceso de limpieza.

In [7]:
df.isnull().mean() * 100

id                0.000000
url               0.000000
region            0.000000
region_url        0.000000
price             0.000000
year              0.282281
manufacturer      4.133714
model             1.236179
condition        40.785232
cylinders        41.622470
fuel              0.705819
odometer          1.030735
title_status      1.930753
transmission      0.598763
VIN              37.725356
drive            30.586347
size             71.767476
type             21.752717
paint_color      30.501078
image_url         0.015930
description       0.016398
county          100.000000
state             0.000000
lat               1.534155
long              1.534155
posting_date      0.015930
dtype: float64

**Interpretación de Valores Nulos**

1. **Variables Completas (0% faltantes)**

    - `id` y `url` (0%): Son identificadores únicos del sistema. Su completitud del 100% indica que la base de datos mantiene integridad en su estructura fundamental. Esto es crítico para trazar cada registro individualmente.

    - `region` y `region_url` (0%): La ubicación geográfica principal está siempre registrada. Esto sugiere que el proceso de recolección valora y exige esta información, probablemente porque es esencial para el negocio o análisis de mercado regional.

    - `price` (0%): La variable objetivo principal está completa. Para un modelo de predicción de precios, esto es excelente. Indica que el precio no es opcional en los listados, lo cual tiene sentido comercialmente.

    - `state` (0%): Complementa la información regional, proporcionando contexto geográfico completo.

2. **Variables con Faltantes Leves (<5%)**

    - `year` (0.28%): Esto indica que el año del vehículo es considerado información casi obligatoria, probablemente porque afecta significativamente el valor.

    - `odometer` (1.03%): El kilometraje está casi completo.

    - `transmission` (0.60%) y `fuel` (0.71%): Características técnicas básicas están muy completas. Los faltantes mínimos sugieren que son campos casi obligatorios en el formulario.

3. **Variables con Faltantes Moderados (5-30%)**

    - `manufacturer` (4.13%): Aproximadamente en 4 de cada 100 vehículos no se registró la marca.

    - `model` (1.24%): Interesantemente, hay más marcas faltantes que modelos. 

    - `title_status` (1.93%): Esto indica que es información generalmente disponible pero no estrictamente obligatoria.

4.  **Variables con Faltantes Críticos (30-70%)**

    - `condition` (40.79%): Esto es alarmante porque la condición es crucial para la valoración. Sugiere que muchos vendedores omiten esta información o el campo no es obligatorio.

    - `cylinders` (41.62%): Casi la mitad de los vehículos no especifican cilindrada. Podría deberse a que muchos vendedores no conocen esta información técnica.

    - `VIN` (37.73%): Más de un tercio no tiene VIN. Esto es problemático para verificación de historial. Podría indicar vehículos más antiguos o vendedores que protegen privacidad.

    - `drive` (30.59%): Información técnica que muchos omiten.

    - `paint_color` (30.50%): Casi un tercio no especifica color.

5. **Variable con Faltantes Extremos (>70%)**

    - `size` (71.77%): Casi 3 de cada 4 vehículos no tienen tamaño especificado. Esto indica que este campo probablemente no existía en formularios antiguos o fue añadido posteriormente.

    - `county` (100%): Variable completamente inútil. Debe eliminarse inmediatamente. Su completa ausencia sugiere que era un campo planeado pero nunca implementado.

El análisis revela un dataset con integridad fundamental en variables críticas como precio, ubicación e identificadores, pero con debilidades significativas en atributos descriptivos como condición, cilindrada y características técnicas. La presencia masiva de valores faltantes en variables clave sugiere problemas estructurales en la recolección de datos más que errores aleatorios.


# Preprocesamiento

Este proceso incluye la limpieza de datos, que implica identificar y corregir errores e inconsistencias, así como la transformación de los datos para prepararlos para el análisis. Es esencial para garantizar que los datos estén listos para el modelado e interpretación, mejorando la calidad y precisión de los resultados. Para garantizar la calidad y manejabilidad del conjunto de datos, se tomaron las siguientes decisiones de limpieza inicial: eliminación de variables y eliminación de observaciones.

## Eliminación de Variables

Se descartaron las columnas `['county', 'id', 'url', 'region_url', 'VIN', 'image_url', 'description', 'long', 'lat', 'posting_date']` por las siguientes razones:

- `id`, `url`, `region_url`, `image_url`: Son identificadores o enlaces únicos que no aportan valor predictivo a un modelo de machine learning.

- `VIN`: El Vehicle Identification Number es un identificador que no aporta valor predictivo a un modelo de machine learning.

- `description`: Es un campo de texto libre. Su procesamiento requeriría técnicas de NLP (Procesamiento de Lenguaje Natural), lo cual no está contemplado en los objetivos de este análisis.

- `long` y `lat` (coordenadas geográficas): Si bien la ubicación es un factor crucial para el precio y la demanda, su tratamiento efectivo requiere de análisis geoespacial avanzado. Dado que el proyecto se centra en la aplicación de modelos de regresión lineal y logística, y para mantener un enfoque claro y reproducible, se decidió excluir estas variables, dejando su potencial análisis para un proyecto futuro.

- `posting_date`: La variable temporal es extremadamente valiosa para analizar tendencias y estacionalidad. Sin embargo, su incorporación adecuada requiere un feature engineering específico (como extraer el día de la semana, el mes, o si es fin de semana). Al igual que con las coordenadas, se optó por excluirla para priorizar el alcance definido, reconociendo su potencial para un análisis más profundo posterior.

- `county`: Variable completamente inútil ya que esta totalmente vacía.

In [8]:
df = df.drop(columns=['county', 'id', 'url', 'region_url', 'VIN', 'image_url', 'description', 'long', 'lat', 'posting_date'])

## Eliminación de Observaciones con Valores Faltantes (NA)

Se optó por eliminar todas las filas que contuvieran al menos un valor NA. Esta decisión se basó en dos factores clave:

- El proyecto sugiere un tamaño típico del conjunto de datos final entre 15.000 y 30.000 observaciones. El dataset original, con cientos de miles de entradas, permite esta reducción sin comprometer la capacidad de los modelos para generalizar.

- Un análisis preliminar reveló un alto porcentaje de valores nulos en varias columnas clave. Realizar una imputación robusta en tantas variables habría introducido un alto grado de incertidumbre y potencial sesgo en los datos, arriesgando la integridad del análisis. La eliminación garantiza la consistencia y calidad de los puntos de datos utilizados para el entrenamiento, resultando en un dataset más pequeño pero significativamente más limpio y confiable.

In [9]:
df = df.dropna() 

---

Se verifican los cambios realizados.

In [10]:
df.shape

(79195, 16)

## Valores únicos

Se calcula el número de valores únicos para cada variable del DataFrame.

In [11]:
df.nunique()

region            403
price            4241
year              102
manufacturer       41
model            9530
condition           6
cylinders           8
fuel                5
odometer        28024
title_status        6
transmission        3
drive               3
size                4
type               13
paint_color        12
state              51
dtype: int64

**Interpretación Valores únicos**

El análisis de cardinalidad de las variables categóricas y numéricas revela características fundamentales sobre la estructura y complejidad del dataset, cruciales para el diseño del pipeline de preprocesamiento. Se observa una cardinalidad extremadamente alta en la variable `model` (9,530 valores únicos), lo que presenta un desafío significativo para técnicas de codificación como One-Hot Encoding, ya que generaría una matriz de features excesivamente amplia y dispersa, incrementando la dimensionalidad y el riesgo de sobreajuste. La variable `manufacturer` (41 únicos) y `region` (403 únicos) también muestran una cardinalidad media-alta, manejable pero que requerirá una cuidadosa consideración. Por otro lado, variables como `transmission` (3), `fuel` (5), `title_status` (6), `drive` (3), y `condition` (6) exhiben una baja cardinalidad, lo que las hace ideales para ser codificadas sin mayor complicación. La presencia de 51 valores únicos en `state` confirma su uso como variable categórica geográfica. Para las variables numéricas, el alto número de valores únicos en `year`, `odometer`, y `price` es el comportamiento esperado y confirma su naturaleza continua, validando la decisión de escalarlas. Este perfil de cardinalidad sugiere la potencial necesidad de estrategias adicionales para variables como `model`, como agrupar modelos poco frecuentes en una categoría "Other" o utilizar técnicas de encoding alternativas, para garantizar la eficiencia y efectividad de los modelos de machine learning.

## Análisis de Variables Categóricas de Baja y Media Cardinalidad 

Antes de proceder con la codificación de las variables categóricas, es fundamental realizar una inspección de la calidad de los datos en estas columnas. El siguiente análisis de frecuencia para un subconjunto de variables de cardinalidad baja y media tiene un doble propósito:

1.  **Validar la Integridad de los Datos:** Detectar la posible presencia de errores de digitación, inconsistencia en el uso de mayúsculas/minúsculas, o categorías que representen lo mismo con nombres ligeramente diferentes (por ejemplo, "automatic", "Automatic", "auto"). Este tipo de errores crearían categorías artificiales y falsas, perjudicando el aprendizaje del modelo.

2.  **Confirmar la Cardinalidad:** Verificar que el número de categorías únicas para cada variable coincide con el análisis previo y es lógico según el dominio del problema. Esto asegura que el One-Hot Encoding se aplicará sobre un conjunto de categorías limpio y consistente.

Este paso de limpieza es crucial para garantizar que las transformaciones posteriores reflejen fielmente la información contenida en los datos y no amplifiquen artefactos o ruido proveniente de una entrada imperfecta.

In [12]:
cardinalidad_baja = [
    'manufacturer', 'condition', 'cylinders', 'fuel', 
    'title_status', 'transmission', 'drive', 'size', 'type', 'paint_color', 'state'
]

for col in cardinalidad_baja:
    print(f"\n=== {col.upper()} ===")
    print(df[col].value_counts())
    print("-" * 50)


=== MANUFACTURER ===
manufacturer
ford               15515
chevrolet          12053
toyota              7005
honda               4853
nissan              3992
jeep                3213
gmc                 3204
ram                 2881
dodge               2531
bmw                 2194
subaru              1976
mercedes-benz       1917
hyundai             1751
volkswagen          1745
kia                 1407
chrysler            1359
lexus               1236
cadillac            1211
buick               1038
mazda                988
audi                 932
acura                721
pontiac              719
lincoln              671
infiniti             648
mitsubishi           612
volvo                607
mini                 420
mercury              400
saturn               342
rover                318
porsche              262
jaguar               216
fiat                 127
tesla                 34
alfa-romeo            28
harley-davidson       26
ferrari               16
datsun         

## Creación de la Variable Binaria "HighDemand"

Para la tarea de clasificación binaria especificada en los objetivos del proyecto, se creó la variable objetivo `HighDemand`.

In [13]:
df['HighDemand'] = (df['price'] > df['price'].median()).astype(int)

Se verifican los cambios realizados.

In [14]:
df.shape

(79195, 17)

## Definición del Esquema de Preprocesamiento para el Pipeline

Para configurar el `ColumnTransformer` que integrará el pipeline de machine learning, es necesario definir de manera explícita qué columnas del DataFrame serán tratadas como variables categóricas y cuáles como numéricas. Esta segmentación es crucial para aplicar las transformaciones adecuadas a cada tipo de dato.

El siguiente código identifica automáticamente todas las columnas de tipo objeto (strings) como categóricas, mientras que las variables numéricas ['year', 'odometer'] se especifican de forma manual. Esta decisión se toma porque, aunque el DataFrame podría contener otras columnas numéricas, solo estas dos son relevantes y requieren ser escaladas para el modelo, ya que variables como `'price'` o `'HighDemand'` son las variables objetivo y no deben ser incluidas como features.

In [15]:
categorical_cols = df.select_dtypes(include=['object']).columns

numeric_cols = ["year", "odometer"]

X = df[categorical_cols.tolist() + numeric_cols]
y_reg = df['price']  
y_clf = df['HighDemand']  

## Análisis de Relevancia de Variables Categóricas (Prueba de Kruskal-Wallis)

Dado que el análisis de ANOVA paramétrico se realizó y arrojó valores de F-statistic extremadamente bajos y p-values altos, se sospechó que los supuestos requeridos para este test (normalidad de residuos y homocedasticidad) podrían no cumplirse, posiblemente debido a la presencia de outliers o a una distribución no normal de la variable objetivo price. Para validar esta hipótesis y obtener una medida más robusta de la relevancia de las variables categóricas, se aplicó la prueba de Kruskal-Wallis.

Esta prueba es la alternativa no paramétrica al ANOVA de una vía. No asume normalidad en los datos y es menos sensible a outliers, ya que trabaja con los rangos de los datos en lugar de sus valores absolutos. Evalúa la hipótesis nula de que las medianas de price son iguales entre todas las categorías de una variable.

- H₀: Las medianas del precio son iguales en todos los grupos.

- H₁: Al menos un grupo tiene una mediana del precio diferente.

El estadístico H obtenido indica la magnitud de la diferencia entre los grupos, donde un valor más alto sugiere una variable más relevante para predecir el precio.

In [16]:
resultados = {}

for col in categorical_cols:
    grupos = [y_reg[X[col] == cat] for cat in X[col].unique()]
    
    if len(grupos) > 1:
        stat, p = kruskal(*grupos)
        resultados[col] = {"H": stat, "p_value": p}

kruskal_df = pd.DataFrame(resultados).T.sort_values("H", ascending=False)

print(kruskal_df)

                         H        p_value
model         36024.049350   0.000000e+00
type          10962.040912   0.000000e+00
drive          8037.182863   0.000000e+00
region         7052.036645   0.000000e+00
condition      6961.989182   0.000000e+00
manufacturer   6244.056408   0.000000e+00
cylinders      5280.314894   0.000000e+00
fuel           4720.109401   0.000000e+00
size           4490.845603   0.000000e+00
state          3280.747507   0.000000e+00
paint_color    2461.664413   0.000000e+00
title_status    737.485363  3.848597e-157
transmission    428.379248   9.519810e-94


## Análisis de Correlación No Paramétrica para Variables Numéricas (Spearman)

Tras los resultados obtenidos en los análisis anteriores, se evidenció la necesidad de emplear métodos estadísticos robustos que no dependieran de supuestos restrictivos como la normalidad de los datos o la linealidad perfecta de las relaciones. Para evaluar la asociación entre las variables numéricas predictoras (year, odometer) y la variable objetivo price, se utilizó el coeficiente de correlación de Spearman.

A diferencia del coeficiente de Pearson, que mide únicamente la relación lineal, el coeficiente de Spearman evalúa relaciones monótonas (ya sean lineales o no lineales, siempre que la tendencia sea constante). Además, al basarse en los rangos de los datos en lugar de sus valores brutos, es mucho menos sensible a outliers, lo que lo hace ideal para analizar variables como el precio, que suelen presentar valores extremos y distribuciones sesgadas.


In [17]:
resultados = {}

for col in numeric_cols:
    corr, p = spearmanr(X[col], y_reg)
    resultados[col] = {"Corr": corr, "p_value": p}

spearman_df = pd.DataFrame(resultados).T.sort_values("Corr", ascending=False)

spearman_df

,Corr,p_value
year,0.438977,0.0
odometer,-0.396387,0.0


## Reducción de Dimensionalidad mediante Eliminación de Variables de Baja Relevancia

Con el objetivo de simplificar el modelo final, mejorar su generalización y reducir el costo computacional, se realizó una selección de variables basada en los resultados del análisis estadístico previo. Las pruebas de Kruskal-Wallis proporcionaron una medida robusta (el estadístico H) de la capacidad de cada variable categórica para discriminar entre diferentes precios.

Se estableció como criterio de eliminación aquellas variables que mostraron un estadístico H bajo y por lo tanto, no aportaban información predictiva valuable al modelo. La eliminación de estas variables irrelevantes ayuda a mitigar el sobreajuste (overfitting), especialmente importante en modelos lineales con alta dimensionalidad por el uso de One-Hot Encoding.

- `paint_color`: El color del vehículo demostró tener una influencia mínima y no consistente en el precio, según el análisis no paramétrico.

- `title_status`: El estado del título del vehículo no mostró diferencias significativas en las medianas de precio entre sus categorías para este conjunto de datos.

- `transmission`: El tipo de transmisión presentó un poder discriminatorio muy bajo en el precio, debido a que su valor ya está correlacionado o capturado por otras variables como el año o el modelo.

In [ ]:
df = df.drop(columns=['paint_color', 'title_status', 'transmission'])

Se verifican los cambios realizados.

In [19]:
df.shape

(79195, 14)

## Agrupamiento de Variables Categóricas de Alta Cardinalidad

En el análisis de datos, las variables categóricas con un gran número de categorías o niveles —conocidas como variables de alta cardinalidad— representan un desafío importante. Estas variables pueden dificultar la construcción de modelos debido al exceso de niveles poco representativos, la presencia de categorías con muy baja frecuencia y el aumento de la dimensionalidad al aplicar técnicas de codificación. Para superar este problema, una estrategia común es el agrupamiento de categorías, que consiste en combinar o reducir los niveles de la variable de manera que se conserven patrones relevantes y se minimice la pérdida de información. Este proceso permite simplificar la estructura de los datos, mejorar la eficiencia de los algoritmos de modelado y, en muchos casos, aumentar la capacidad predictiva de los modelos al evitar el sobreajuste.

### Agrupamiento por Frecuencia para la variable "manufacturer"

Para manejar la alta cardinalidad en la variable `manufacturer` (41 categorías originales) y evitar alta dimensionalidad en los modelos de machine learning, se implementó una estrategia de agrupamiento basada en frecuencia. Se estableció un umbral absoluto de 1000 observaciones, donde todos los fabricantes con una frecuencia menor a este valor fueron agrupados en una nueva categoría denominada `'other_manufacturer'`.

- **Simplicidad y objetividad:** El criterio basado únicamente en frecuencia es reproducible y evita sesgos subjetivos.
- **Suficiencia de datos:** Las marcas preservadas (>1000 observaciones) cuentan con volumen suficiente para que los modelos identifiquen patrones consistentes en sus precios.
- **Manejo del largo tail:** La estrategia agrupa eficientemente más de 20 marcas de baja frecuencia que individualmente aportaban ruido estadístico al modelo.

**Resultado obtenido:** Esta intervención redujo la cardinalidad de la variable, manteniendo la información de las marcas más representativas del mercado mientras se optimiza la eficiencia computacional.

---

**Alternativa Considerada: Estrategia Híbrida (Frecuencia + Valor de Marca)**

Se contempló una estrategia alternativa más sofisticada que combinara el umbral de frecuencia con el conocimiento del dominio automotriz:

**Criterio propuesto:** Aplicar un umbral absoluto bajo, pero preservar manualmente marcas icónicas/lujosas sin importar su volumen.

**Ventajas de la alternativa:**
- Captura el efecto de marcas de lujo que, debido a su alto precio presentan menos frecuencias en el Dataset.
- Representa la opción más inteligente desde una perspectiva del dominio del problema automotriz.
- Podría mejorar la capacidad predictiva del modelo para vehículos de gama alta.

**Limitaciones que motivaron su descarte:**
- Requiere conocimiento experto del dominio o investigación adicional para definir consistentemente qué marcas se consideran "premium".
- Introduce un elemento de subjetividad en el preprocesamiento de datos.
- Aumenta la complejidad del pipeline al necesitar un mapeo manual personalizado.
- Extiende el número de categorías finales, incrementando ligeramente la dimensionalidad.

**Decisión final:** Se optó por la estrategia basada puramente en frecuencia por su mayor objetividad, reproducibilidad y alineación con el principio de simplicidad en la etapa inicial del proyecto. La alternativa híbrida se identifica como una mejora potencial para iteraciones futuras una vez validado el desempeño del enfoque actual.


In [20]:
umbral = 1000

fabricantes_a_agrupar = df['manufacturer'].value_counts()[df['manufacturer'].value_counts() < umbral].index.tolist()

df['manufacturer'] = df['manufacturer'].apply(lambda x: 'other_manufacturer' if x in fabricantes_a_agrupar else x)


### Agrupamiento por Frecuencia para la variable "state"

Para optimizar el rendimiento del modelo y manejar la distribución desigual de datos geográficos en la variable `state` (51 categorías originales correspondientes a los estados de EE.UU.), se aplicó una estrategia de agrupamiento basada en frecuencia utilizando con el mismo umbral absoluto de 1000 observaciones establecido para los fabricantes.

- **Consistencia metodológica:** Mantener el mismo umbral en variables similares asegura coherencia en el preprocesamiento y facilita la reproducibilidad del análisis.
- **Manejo de desbalanceo geográfico:** El dataset presenta una distribución heterogénea entre estados, donde mercados grandes (ej: California, Florida) concentran la mayoría de observaciones mientras estados menos poblados tienen representación mínima.
- **Prevención de sobreajuste:** Estados con muy pocas observaciones (<1000) pueden introducir ruido y patrones espurios que afectan la generalización del modelo.

**Impacto obtenido:** Esta intervención redujo significativamente la cardinalidad de la variable, agrupando estados de baja densidad demográfica o con menor actividad en el mercado de autos usados en la categoría `'other_state'`, mientras se preservaron los mercados estatales más relevantes para el análisis.

In [21]:
estados_a_agrupar = df['state'].value_counts()[df['state'].value_counts() < umbral].index.tolist()

df['state'] = df['state'].apply(lambda x: 'other_state' if x in estados_a_agrupar else x)


### Agrupamiento por Frecuencia para la variable "region"

Para abordar la extrema cardinalidad de la variable `region` (403 categorías originales) y prevenir severos problemas de dimensionalidad en el modelo, se implementó una estrategia de agrupamiento utilizando un umbral absoluto de 100 observaciones. Todas las regiones con una frecuencia menor a este valor fueron consolidadas en una categoría única denominada `'other_region'`.

- **Cardinalidad crítica:** La presencia de 403 categorías representaba el mayor desafío de dimensionalidad en el dataset, haciendo esencial una reducción drástica.
- **Naturaleza de los datos:** La distribución de regiones mostró una fuerte concentración en mercados metropolitanos principales y una larga cola de regiones con representación mínima.
- **Prevención de sobreajuste extremo:** Regiones con menos de 100 observaciones carecen del volumen necesario para que el modelo aprenda patrones confiables, representando principalmente ruido estadístico.

**Impacto obtenido:** Esta intervención redujo la cardinalidad de la variable, eliminando eficientemente regiones que contribuían marginalmente al volumen total de datos pero representaban un riesgo significativo de sobreajuste.

In [22]:
umbral = 100

regiones_a_agrupar = df['region'].value_counts()[df['region'].value_counts() < umbral].index.tolist()

df['region'] = df['region'].apply(lambda x: 'other_region' if x in regiones_a_agrupar else x)


### Preservación Selectiva de Modelos por Frecuencia Intra-Marca

Para abordar el desafío más crítico de cardinalidad en el dataset -la variable `model` con 9,530 categorías únicas- se implementó una estrategia sofisticada de agrupamiento que preserva los modelos más significativos dentro del contexto de cada fabricante. La metodología consistió en mantener los 10 modelos más frecuentes por marca y agrupar el resto bajo la denominación `"other_{marca}"`.

- **Cardinalidad extrema:** La variable representaba el mayor riesgo de sobreajuste por dimensionalidad, haciendo esencial una reducción drástica pero inteligente.
- **Relevancia contextual:** Un modelo solo tiene significado predictivo en el contexto de su marca (ej: "camry" es relevante para Toyota pero no existe en Ford).
- **Preservación de valor:** Mantener los modelos más populares de cada marca captura el efecto de los vehículos icónicos mientras elimina el ruido de modelos raros o especializados.
- **Balance óptimo:** La selección de 10 modelos por marca representa un punto óptimo entre retener información valiosa y controlar la dimensionalidad.

**Impacto obtenido:** Esta intervención redujo la cardinalidad a un número manejable, transformando una variable inutilizable en una característica viable para el modelado predictivo.

---

**Alternativa Considerada: Eliminación Pura de la Variable**

Se evaluó la opción más conservadora de eliminar completamente la variable del modelo:

**Criterio propuesto:** "Eliminar la variable `model` y confiar en que las demás características (`manufacturer`, `type`, `year`, etc.) capturen suficiente varianza explicativa."

**Ventajas de la alternativa:**
- Eliminación completa del problema de cardinalidad
- Simplificación radical del pipeline de preprocesamiento
- Reducción del riesgo de sobreajuste por variables correlacionadas

**Limitaciones que motivaron su descarte:**
- Pérdida irreversible de información valiosa sobre modelos específicos que afectan significativamente el precio
- Incapacidad para capturar el efecto de modelos icónicos dentro de una marca
- Reducción potencial del poder predictivo del modelo

**Decisión final:** Se optó por la estrategia de agrupamiento intra-marca por su capacidad para transformar una variable problemática en un recurso valioso. Este enfoque demuestra cómo el conocimiento del dominio automotriz (reconocer que la importancia de un modelo depende de su marca) puede informar técnicamente las decisiones de preprocesamiento. La solución implementada representa el balance ideal entre sofisticación técnica y pragmatismo aplicado.

**Nota:** La estrategia se aplicó consistentemente incluso a la categoría `'other_manufacturer'`, donde se identificaron los 10 modelos más frecuentes dentro de este grupo consolidado, asegurando integridad metodológica en todo el dataset.


In [23]:

top_models_por_marca = {}

for marca in df['manufacturer'].unique():

    df_marca = df[df['manufacturer'] == marca]
    
    top_models = df_marca['model'].value_counts().head(10).index.tolist()
    
    top_models_por_marca[marca] = top_models


df['model'] = df.apply(
    lambda row: row['model'] if row['model'] in top_models_por_marca[row['manufacturer']] 
    else f"other_{row['manufacturer']}", 
    axis=1
)


Se muestran las primeras filas del DataFrame con todos los detalles.  
Se usa `.to_string()` para evitar el truncamiento de columnas y lograr mostrar todas las variables.

In [24]:
print(df.head().to_string())

          region  price    year manufacturer        model  condition    cylinders    fuel  odometer drive       size      type        state  HighDemand
31  other_region  15000  2013.0         ford   other_ford  excellent  6 cylinders     gas  128000.0   rwd  full-size     truck  other_state           1
55  other_region  19900  2004.0         ford   other_ford       good  8 cylinders  diesel   88000.0   4wd  full-size    pickup  other_state           1
59  other_region  14000  2012.0        honda      odyssey  excellent  6 cylinders     gas   95000.0   fwd  full-size  mini-van  other_state           1
65  other_region  22500  2001.0         ford   other_ford       good  8 cylinders  diesel  144700.0   rwd  full-size     truck  other_state           1
73  other_region  15000  2017.0        dodge  other_dodge  excellent  8 cylinders     gas   90000.0   rwd   mid-size     sedan  other_state           1



Se muestran las últimas filas del DataFrame con todos los detalles.  
Se usa `.to_string()` para evitar el truncamiento de columnas y lograr mostrar todas las variables.

In [25]:
print(df.tail().to_string())

         region  price    year        manufacturer                     model  condition    cylinders    fuel  odometer drive         size         type        state  HighDemand
426808  wyoming  18000  2005.0           chevrolet           other_chevrolet  excellent  8 cylinders     gas  130000.0   4wd    full-size        truck  other_state           1
426809  wyoming  18000  1990.0                jeep                other_jeep       good  8 cylinders     gas  114400.0   4wd    full-size          SUV  other_state           1
426831  wyoming   9800  1985.0              nissan              other_nissan   like new  6 cylinders     gas  115000.0   rwd  sub-compact    hatchback  other_state           1
426833  wyoming   6800  1997.0  other_manufacturer  other_other_manufacturer       good  8 cylinders     gas   69550.0   rwd      compact  convertible  other_state           0
426836  wyoming      0  2018.0                 ram                      2500  excellent  6 cylinders  diesel   20492.0  

# Construcción de Pipelines

La construcción de pipelines constituye una herramienta fundamental para garantizar procesos de análisis ordenados, reproducibles y eficientes. Un pipeline permite encadenar de manera estructurada todas las etapas del flujo de trabajo, desde la preparación de los datos (limpieza, transformación y selección de variables) hasta la aplicación de algoritmos de modelado y la evaluación de resultados. Esta integración facilita la automatización, reduce el riesgo de errores y asegura que el mismo procedimiento pueda aplicarse a nuevos conjuntos de datos con consistencia. Además, los pipelines son especialmente útiles cuando se combinan con técnicas de validación cruzada y búsqueda de hiperparámetros, ya que permiten optimizar el desempeño de los modelos dentro de un marco sistemático y escalable.

## Construcción del Preprocesador Central con ColumnTransformer

Para integrar de manera segura y eficiente todas las transformaciones de preprocesamiento en un flujo único y reproducible, se construye un `ColumnTransformer`. Este objeto es el núcleo del pipeline, ya que permite aplicar diferentes transformaciones a subconjuntos específicos de columnas, garantizando que cada paso se ejecute en el orden correcto y sin fugas de información (data leakage) durante la validación cruzada.

La configuración del preprocesador especifica dos transformaciones en paralelo:

1. **Escalado estandarizado** para variables numéricas continuas.

2. **Codificación one-hot** con eliminación de la primera categoría para evitar multicolinealidad en las variables categóricas.



In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),  
        ('cat', OneHotEncoder(drop='first'), categorical_cols)  
    ]
)

## Construcción de Pipelines para Modelos de Regresión y Clasificación

Para cumplir con el objetivo de integrar un flujo completo de machine learning, se construyen pipelines independientes para los algoritmos de regresión Ridge y Lasso y clasificación (Regresión Logística). Un pipeline encapsula de manera secuencial y automatizada todas las etapas del proceso de modelado, desde el preprocesamiento de los datos hasta la ejecución del algoritmo de aprendizaje. Esto garantiza que cada transformación se aplique de manera consistente durante el entrenamiento y la predicción, eliminando el riesgo de data leakage y asegurando la reproducibilidad de los resultados.

Cada pipeline se compone de dos etapas fundamentales:

1. `preprocessor`: La transformación de características definida previamente en el ColumnTransformer, que se encarga de escalar las variables numéricas y codificar las categóricas.

2. `model`: El estimador o algoritmo de machine learning que aprenderá a predecir el precio a partir de las características transformadas.

La elección de estos modelos específicos permite evaluar diferentes enfoques:

- **Ridge y Lasso**: Permiten evaluar el impacto de dos tipos diferentes de regularización (L2 y L1, respectivamente) en el rendimiento para la tarea de regresión (predecir el precio).

- **Regresión Logística**: Proporciona un modelo lineal base para la tarea de clasificación binaria (predecir la demanda alta o baja), cuya optimización se controla con el hiperparámetro de regularización `C`.


In [27]:
#(regularización L2)
pipeline_ridge = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

#(regularización L1)
pipeline_lasso = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Lasso())
])

pipeline_logreg = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])


# Entrenamiento y Ajuste

Durante el entrenamiento, el algoritmo ajusta sus parámetros internos en función de los ejemplos disponibles, minimizando el error entre las predicciones y los valores reales. Posteriormente, el ajuste implica la selección de los mejores hiperparámetros mediante técnicas como la validación cruzada o la búsqueda en malla, con el fin de mejorar la capacidad predictiva del modelo y evitar problemas como el sobreajuste o el subajuste. Este proceso garantiza que el modelo no solo funcione adecuadamente con los datos de entrenamiento, sino que también tenga un buen desempeño al generalizar en nuevos escenarios.

## Configuración de la Búsqueda de Hiperparámetros con GridSearchCV

Para optimizar el rendimiento de los modelos y encontrar la mejor combinación de hiperparámetros, se configura una búsqueda exhaustiva mediante `GridSearchCV`. Esta estrategia prueba de manera sistemática todas las combinaciones de los valores especificados, utilizando validación cruzada para evaluar de forma robusta el desempeño de cada configuración. La configuración se realiza de manera independiente para cada pipeline, definiendo la estrategia de validación cruzada y la métrica de evaluación apropiadas para cada tarea (regresión o clasificación).

Se implementan dos estrategias de validación cruzada:

- `KFold` para los problemas de regresión, que divide los datos en k folds aleatorios preservando el tamaño de los conjuntos.

- `StratifiedKFold` para el problema de clasificación, que garantiza que cada fold tenga aproximadamente la misma proporción de la variable objetivo, crucial para conjuntos desbalanceados.

Además, se define la métrica de evaluación para cada tarea:

- `neg_mean_squared_error` para regresión, donde un valor más cercano a cero indica mejor rendimiento.

- `accuracy` para clasificación, que mide la proporción de predicciones correctas.

Los hiperparámetros a optimizar son:

- `alpha` para Ridge y Lasso, que controla la fuerza de la regularización.

- `C` para Regresión Logística, que controla la inversa de la fuerza de la regularización.

In [28]:

cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_ridge = GridSearchCV(
    estimator=pipeline_ridge,
    param_grid={'model__alpha': [0.01, 0.1, 1, 10, 100]},
    cv=cv_reg,
    scoring='neg_mean_squared_error'
)

grid_lasso = GridSearchCV(
    estimator=pipeline_lasso,
    param_grid={'model__alpha': [0.01, 0.1, 1, 10, 100]},
    cv=cv_reg,
    scoring='neg_mean_squared_error'
)

grid_logreg = GridSearchCV(
    estimator=pipeline_logreg,
    param_grid={'model__C': [0.01, 0.1, 1, 10, 100]},
    cv=cv_clf,
    scoring='accuracy'
)


## Ejecución de la Búsqueda de Hiperparámetros y Entrenamiento de los Modelos

Una vez configurados los pipelines y las estrategias de búsqueda, se procede a ejecutar el entrenamiento de los modelos. Este paso es donde se implementa de manera práctica el flujo completo de machine learning definido teóricamente. El método `fit()` de cada objeto `GridSearchCV` no solo entrena el modelo final con la mejor combinación de hiperparámetros encontrada, sino que realiza de manera automatizada y segura todo el proceso de validación cruzada, preprocesamiento y optimización.

In [29]:

grid_ridge.fit(X, y_reg)

GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['year',
                                                                          'odometer']),
                                                                        ('cat',
                                                                         OneHotEncoder(drop='first'),
                                                                         Index(['region', 'manufacturer', 'model', 'condition', 'cylinders', 'fuel',
       'title_status', 'transmission', 'drive', 'size', 'type', 'paint_color',
       'state'],
      dtype='object'))])),
                                       ('model', Ridge())]),
             param_grid={'model__alpha': [0.01, 0.1, 1, 10, 100]},
             scoring='neg_mean_squared_error')

In [ ]:
grid_lasso.fit(X, y_reg)

In [ ]:
grid_logreg.fit(X, y_clf)

## Análisis de los Resultados de la Búsqueda de Hiperparámetros

Tras completar el proceso de búsqueda exhaustiva, se extraen y analizan los resultados óptimos para cada modelo. Esta etapa es crucial para identificar la configuración de hiperparámetros que maximiza el rendimiento de cada algoritmo según la métrica de evaluación definida. Los resultados se presentan mostrando la mejor combinación de parámetros encontrada y su score de validación cruzada correspondiente.

Es importante notar que para los modelos de regresión (Ridge y Lasso), el score reportado por `GridSearchCV` es el negative mean squared error (`neg_mean_squared_error`). Por convención de scikit-learn, este valor se negativiza para que el algoritmo de optimización siempre busque maximizar el score. Por lo tanto, para interpretar el resultado en la escala original del Error Cuadrático Medio (MSE), es necesario multiplicar el valor por -1, obteniendo así el MSE promedio en la validación cruzada. Para el modelo de clasificación (Regresión Logística), el score de precisión (accuracy) se reporta directamente en su escala interpretable.

In [ ]:
print("Mejor Ridge:", grid_ridge.best_params_, "Score (MSE):", -grid_ridge.best_score_)
print("Mejor Lasso:", grid_lasso.best_params_, "Score (MSE):", -grid_lasso.best_score_)
print("Mejor LogReg:", grid_logreg.best_params_, "Score (Accuracy):", grid_logreg.best_score_)


# Evaluación de desempeño

La evaluación de desempeño es una etapa fundamental en el desarrollo de modelos de machine learning, ya que permite medir qué tan bien se ajusta el modelo a los datos y, sobre todo, qué capacidad tiene para generalizar en situaciones no vistas. A través de métricas específicas (que varían según si se trata de un problema de regresión o de clasificación) es posible identificar fortalezas y debilidades del modelo, así como comparar diferentes enfoques y versiones durante el proceso de experimentación. Además de las métricas numéricas, las visualizaciones como matrices de confusión, curvas ROC o gráficos de predicciones frente a valores reales ofrecen una perspectiva más clara sobre el rendimiento alcanzado. De esta manera, la evaluación del desempeño no solo valida la efectividad del modelo, sino que también orienta la toma de decisiones para su mejora continua.

## Creación de Conjuntos de Entrenamiento y Prueba

Para evaluar de manera rigurosa y justa el desempeño final de los modelos optimizados, es esencial probarlos en datos que no hayan sido vistos durante el proceso de ajuste de hiperparámetros. Por esta razón, se divide el dataset en subconjuntos de entrenamiento y prueba. El conjunto de entrenamiento (`X_train`, `y_train`) se utilizará para reentrenar el mejor modelo encontrado por `GridSearchCV` con todos los datos disponibles de entrenamiento, mientras que el conjunto de prueba (`X_test`, `y_test`) se reservará exclusivamente para la evaluación final, proporcionando una estimación no sesgada del rendimiento del modelo en datos nuevos.

La división se realiza dos veces de manera independiente:

1. Para la tarea de Regresión: Se divide las features (`X`) y la variable objetivo continua (`y_reg`).

2. Para la tarea de Clasificación: Se divide las features (`X`) y la variable objetivo binaria (`y_clf`), utilizando el parámetro `stratify` para garantizar que la proporción de las clases `HighDemand` se mantenga igual en ambos conjuntos, preservando así el balance original.

Un 20% de los datos se asigna para prueba en ambos casos, utilizando una semilla (`random_state=42`) para asegurar la reproducibilidad de la partición.

In [ ]:

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X, y_clf, test_size=0.2, stratify=y_clf, random_state=42
)


## Evaluación Final del Modelo de Regresión en el Conjunto de Prueba

Una vez identificados los mejores hiperparámetros mediante validación cruzada, se procedió a realizar la evaluación final de los modelos optimizados en un conjunto de prueba que no fue utilizado durante ninguna etapa del ajuste o selección del modelo. Este paso es crucial para obtener una estimación realista e imparcial del rendimiento que se podría esperar en datos nuevos y no vistos.

Cada mejor modelo (`best_estimator_`) obtenido del GridSearchCV se reentrenó con la totalidad del conjunto de entrenamiento (`X_train_reg`, `y_train_reg`) para aprovechar toda la información disponible y luego se generaron predicciones sobre el conjunto de prueba reservado (`X_test_reg`). El desempeño se cuantificó utilizando tres métricas de regresión clave:

- **MAE (Error Absoluto Medio)**: Representa el error promedio en las unidades originales de la variable objetivo (dólares), siendo fácil de interpretar.

- **RMSE (Raíz del Error Cuadrático Medio)**: Penaliza más los errores grandes, por lo que es útil para identificar modelos con predicciones muy disparatadas.

- **R² (Coeficiente de Determinación)**: Indica la proporción de la varianza en la variable objetivo que es explicada por el modelo.

Adicionalmente, se generaron gráficos de dispersión de predicciones vs. valores reales para una evaluación visual que permite identificar sesgos, patrones de error y la concordancia general del modelo.

In [ ]:

best_ridge = grid_ridge.best_estimator_
best_ridge.fit(X_train_reg, y_train_reg)

y_pred_reg = best_ridge.predict(X_test_reg)

mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)

print("Evaluación Ridge")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


plt.scatter(y_test_reg, y_pred_reg, alpha=0.6)
plt.plot([y_test_reg.min(), y_test_reg.max()],
         [y_test_reg.min(), y_test_reg.max()],
         'r--', lw=2)
plt.xlabel("Valores reales")
plt.ylabel("Predicciones")
plt.title("Predicciones vs Valores Reales (Regresión)")
plt.show()


best_lasso = grid_lasso.best_estimator_
best_lasso.fit(X_train_reg, y_train_reg) 
y_pred_reg_lasso = best_lasso.predict(X_test_reg)

mae_lasso = mean_absolute_error(y_test_reg, y_pred_reg_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_lasso))
r2_lasso = r2_score(y_test_reg, y_pred_reg_lasso)

print("\nEvaluación Lasso") 
print("MAE:", mae_lasso)
print("RMSE:", rmse_lasso)
print("R²:", r2_lasso)

plt.scatter(y_test_reg, y_pred_reg_lasso, alpha=0.6)
plt.plot([y_test_reg.min(), y_test_reg.max()],
         [y_test_reg.min(), y_test_reg.max()],
         'r--', lw=2)
plt.xlabel("Valores reales")
plt.ylabel("Predicciones")
plt.title("Predicciones vs Valores Reales - Lasso")
plt.show()


## Evaluación Final del Modelo de Clasificación en el Conjunto de Prueba

Para completar el análisis del objetivo de clasificación binaria, se evaluó el desempeño del mejor modelo de Regresión Logística identificado mediante `GridSearchCV` en el conjunto de prueba reservado para esta tarea. La evaluación de un modelo de clasificación requiere un análisis más multidimensional que la regresión, ya que es necesario considerar no solo la capacidad de predicción general (accuracy) sino también el equilibrio entre los diferentes tipos de error (falsos positivos y falsos negativos), lo cual es crucial dependiendo del contexto de aplicación.

El modelo se reentrenó con la totalidad del conjunto de entrenamiento de clasificación (`X_train_clf`, `y_train_clf`) y se generaron tanto predicciones de clase (`predict`) como probabilidades (`predict_proba`) para el conjunto de prueba. El desempeño se evaluó mediante un conjunto exhaustivo de métricas y visualizaciones:

- **Métricas de Evaluación**:

    - **Accuracy**: Proporción general de predicciones correctas.

    - **Precision**: Capacidad del modelo de no etiquetar como de alta demanda a un vehículo que no lo es (evitar falsos positivos).

    - **Recall**: Capacidad del modelo de encontrar todos los vehículos de alta demanda (evitar falsos negativos).

    - **F1-score**: Media armónica entre Precision y Recall, ideal para conjuntos balanceados.

- **Visualizaciones**:

    - **Matriz de Confusión**: Muestra de forma explícita los aciertos (diagonal) y los errores (falsos positivos y falsos negativos) del modelo.

    - **Curva ROC y AUC**: Evalúa la capacidad del modelo para distinguir entre clases a través de todos los posibles umbrales de clasificación. Un AUC de 0.5 representa un modelo aleatorio, mientras que 1.0 representa una separación perfecta.

In [ ]:

best_logreg = grid_logreg.best_estimator_
best_logreg.fit(X_train_clf, y_train_clf)

y_pred_clf = best_logreg.predict(X_test_clf)
y_pred_proba = best_logreg.predict_proba(X_test_clf)[:,1]

acc = accuracy_score(y_test_clf, y_pred_clf)
prec = precision_score(y_test_clf, y_pred_clf)
rec = recall_score(y_test_clf, y_pred_clf)
f1 = f1_score(y_test_clf, y_pred_clf)

print("Evaluación Logistic Regression")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1-score:", f1)

cm = confusion_matrix(y_test_clf, y_pred_clf)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Baja Demanda","Alta Demanda"], yticklabels=["Baja Demanda","Alta Demanda"])
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de confusión")
plt.show()

fpr, tpr, _ = roc_curve(y_test_clf, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'r--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC")
plt.legend(loc="lower right")
plt.show()
